# LangChain LLM Chain Tutorial

This notebook demonstrates the basic concepts of LangChain by building a simple LLM Chain that uses a prompt template and a language model to generate responses.

Based on the LangChain Quickstart tutorial:
https://python.langchain.com/docs/tutorials/llm_chain/

In [ ]:
!pip install langchain-groq python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- Configuración de la API Key ---
# Cargamos la clave desde el archivo .env
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
groq_api_key_found = GROQ_API_KEY is not None

if groq_api_key_found:
    print("Groq API key cargada exitosamente desde el archivo .env")
else:
    print("ADVERTENCIA: GROQ_API_KEY no encontrada. Revisa tu archivo .env")

## 2. Build the LLM Chain

The `build_chain` function creates a simple LangChain LLM Chain. The chain consists of three components connected using the LangChain Expression Language (LCEL) pipe operator (`|`):

1.  `ChatPromptTemplate` – formats the user input into a prompt
2.  `ChatOpenAI` – sends the prompt to the OpenAI API
3.  `StrOutputParser` – extracts the text from the model response

In [ ]:
def build_chain(model_name: str = "llama-3.1-8b-instant", temperature: float = 0.7):
    """
    Build a simple LangChain LLM Chain using a model from Groq.

    Args:
        model_name:  The name of the model to use from Groq.
        temperature: Sampling temperature controlling creativity.

    Returns:
        A runnable LangChain chain.
    """
    # 1. Define the prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un asistente servicial. Responde a la pregunta del usuario de forma clara y concisa."),
        ("human", "{question}"),
    ])

    # 2. Initialise the language model from Groq
    llm = ChatGroq(
        model_name=model_name,
        temperature=temperature,
    )

    # 3. Define an output parser
    output_parser = StrOutputParser()

    # 4. Compose the chain using the LCEL pipe operator
    chain = prompt | llm | output_parser

    return chain

## 3. Use the Chain

The `ask` function invokes the chain with a question and returns the model's response.

In [ ]:
def ask(question: str, chain) -> str:
    """
    Invoke the chain with a question and return the model's response.

    Args:
        question: The question to ask the language model.
        chain:    The LangChain chain returned by build_chain().

    Returns:
        The model's answer as a plain string.
    """
    response = chain.invoke({"question": question})
    return response

## 4. Run the Demo

Now we can build the chain and ask some example questions to see it in action.

In [ ]:
# Build the chain only if the Groq API key is available
if groq_api_key_found:
    # Build the chain once and reuse it for every question
    chain = build_chain()

    # Example questions that showcase the chain
    example_questions = [
        "¿Qué es LangChain y para qué se utiliza?",
        "Explica el concepto de plantillas de prompt en términos sencillos.",
        "¿Qué es el LangChain Expression Language (LCEL)?",
    ]

    for question in example_questions:
        print(f"Pregunta: {question}")
        answer = ask(question, chain)
        print(f"Respuesta:   {answer}")
        print("-" * 60)
else:
    print("Cannot run the demo because the Groq API key is not available.")